# Assignment: DQN for Stock Trading Simulation

**Objective:** Implement a Deep Q-Network (DQN) agent to trade a single stock (e.g., AAPL) over historical daily data, maximizing portfolio value. You'll use raw price windows as states and learn buy/hold/sell actions.

- **States:** Last 30 days' normalized prices + current position (cash/stock held).
- **Actions:** 0=Buy, 1=Sell, 2=Hold.
- **Rewards:** Daily profit/loss; +bonus for positive end portfolio.

We'll use PyTorch for nets and Gymnasium for the env. Focus: Fill blanks for DQN logic (using pseudocode from class). Bonus: Implement Dueling DQN.

**Requirements:**
- Install: `yfinance`, `gymnasium`, `torch`, `numpy`, `matplotlib`.
- Train on 2010–2020 data; test 2021–2023.
- Analyze: Compare to buy-hold baseline.

**Submission:** Completed notebook with plots + answers to questions.

In [ ]:
# Imports (all provided – no blanks here)
import yfinance as yf
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import deque
import random
import matplotlib.pyplot as plt

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## Part 1: Data Loading

Download AAPL historical data. Normalize prices for the agent.

**TODO 1.1:** Fill in the normalization (min-max scale prices).

In [ ]:
# Download data
stock = 'AAPL'
data = yf.download(stock, start='2010-01-01', end='2023-12-31')
prices = data['Close'].values

# Split train/test
split = int(len(prices) * 0.8)  # ~2010-2020 train, 2021-2023 test
train_prices = prices[:split]
test_prices = prices[split:]

# TODO 1.1: Normalize prices to [0,1] (hint: min-max scale)
def normalize(prices):
    min_p = np.min(prices)
    max_p = np.max(prices)
    return _____  # (prices - min_p) / (max_p - min_p)

norm_train = normalize(train_prices)
norm_test = normalize(test_prices)  # Use train min/max for consistency? Or separate? Discuss.

plt.plot(prices)
plt.title('AAPL Close Prices')
plt.show()

## Part 2: Custom Trading Environment

Gym env: Simulates trading over price sequence.

- Window size: 30 days.
- Position: 0 (all cash) to 1 (all stock).

No blanks here – use as is.

In [ ]:
class TradingEnv(gym.Env):
    def __init__(self, prices, window_size=30):
        super().__init__()
        self.prices = prices
        self.window_size = window_size
        self.observation_space = spaces.Box(low=0, high=1, shape=(window_size + 1,))  # prices + position
        self.action_space = spaces.Discrete(3)  # buy, sell, hold
        self.reset()
    
    def reset(self, seed=None, options=None):
        self.current_step = self.window_size
        self.portfolio = 10000.0  # starting cash
        self.position = 0.0  # fraction in stock
        return self._get_obs(), {}
    
    def _get_obs(self):
        window = self.prices[self.current_step - self.window_size:self.current_step]
        return np.append(window, self.position)
    
    def step(self, action):
        current_price = self.prices[self.current_step]
        prev_value = self.portfolio
        
        if action == 0:  # buy
            self.position = min(1.0, self.position + 0.1)  # increment by 10%
        elif action == 1:  # sell
            self.position = max(0.0, self.position - 0.1)
        # hold does nothing
        
        # Update portfolio (assume no fees for simplicity)
        # Avoid div-by-zero when normalized price is 0 (would produce NaN reward)
        next_price = self.prices[self.current_step + 1]
        price_ratio = next_price / max(current_price, 1e-8)
        self.portfolio = self.portfolio * (1 - self.position) + self.position * self.portfolio * price_ratio
        
        reward = self.portfolio - prev_value
        self.current_step += 1
        done = self.current_step >= len(self.prices) - 1
        truncated = False
        
        if done:
            reward += 100 if self.portfolio > 10000 else -100  # end bonus
        
        return self._get_obs(), reward, done, truncated, {}

## Part 3: Replay Buffer

Provided: Deque for storage.

**TODO 3.1:** Fill in the sample method (random batch).

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        # TODO 3.1: Randomly sample batch_size experiences (hint: random.sample)
        batch = _____  # random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return np.array(states), np.array(actions), np.array(rewards), np.array(next_states), np.array(dones)
    
    def __len__(self):
        return len(self.buffer)

## Part 4: Neural Networks

Standard DQN net + Dueling version.

**TODO 4.1:** Fill forward pass for standard net.
**TODO 4.2:** Fill forward for Dueling (V + A).

In [ ]:
class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.fc1 = nn.Linear(state_size, 128)
        self.fc2 = nn.Linear(128, 128)
        self.out = nn.Linear(128, action_size)
    
    def forward(self, x):
        # TODO 4.1: Fill layers (hint: ReLU activations)
        x = _____  # torch.relu(self.fc1(x))
        x = _____  # torch.relu(self.fc2(x))
        return self.out(x)

class DuelingDQN(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.fc1 = nn.Linear(state_size, 128)
        self.fc2 = nn.Linear(128, 128)
        
        # Value stream
        self.value = nn.Linear(128, 1)
        
        # Advantage stream
        self.advantage = nn.Linear(128, action_size)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        
        V = self.value(x)
        A = self.advantage(x)
        
        # TODO 4.2: Combine V and A with mean subtraction (hint: Q = V + (A - A.mean(dim=1, keepdim=True)))
        Q = _____ 
        return Q

## Part 5: DQN Agent

Core logic here.

**TODO 5.1:** Epsilon-greedy action selection.
**TODO 5.2:** Compute targets and loss.
**TODO 5.3:** Soft target update (polyak average, tau=0.005).

In [ ]:
class DQNAgent:
    def __init__(self, state_size, action_size, dueling=False):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = ReplayBuffer(100000)
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.tau = 0.005  # for soft update
        self.batch_size = 64
        self.update_every = 4
        
        model = DuelingDQN if dueling else DQN
        self.qnetwork_local = model(state_size, action_size)
        self.qnetwork_target = model(state_size, action_size)
        self.optimizer = optim.Adam(self.qnetwork_local.parameters(), lr=0.001)
        self.step_count = 0
    
    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0)
        self.qnetwork_local.eval()
        with torch.no_grad():
            action_values = self.qnetwork_local(state)
        self.qnetwork_local.train()
        
        # TODO 5.1: Epsilon-greedy (hint: if random < eps: random action else argmax)
        if random.random() < self.epsilon:
            return _____  # random.randint(0, self.action_size - 1)
        else:
            return _____  # np.argmax(action_values.cpu().data.numpy())
    
    def step(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)
        self.step_count += 1
        
        if self.step_count % self.update_every == 0 and len(self.memory) > self.batch_size:
            self.learn()
        
        if done:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
    
    def learn(self):
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)
        
        states = torch.from_numpy(states).float()
        actions = torch.from_numpy(actions).long().unsqueeze(1)
        rewards = torch.from_numpy(rewards).float()
        next_states = torch.from_numpy(next_states).float()
        dones = torch.from_numpy(dones).float()
        
        # Get Q values
        Q_expected = self.qnetwork_local(states).gather(1, actions).squeeze(1)
        
        # TODO 5.2: Compute targets (hint: for DQN, next_Q = target_net(next_states).max(1)[0]; targets = rewards + gamma * next_Q * (1-dones))
        next_Q = _____  # self.qnetwork_target(next_states).detach().max(1)[0]
        Q_targets = _____  # rewards + (self.gamma * next_Q * (1 - dones))
        
        # Loss (MSE)
        loss = nn.MSELoss()(Q_expected, Q_targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # TODO 5.3: Soft target update (hint: for each param, target = tau * local + (1-tau) * target)
        for target_param, local_param in zip(self.qnetwork_target.parameters(), self.qnetwork_local.parameters()):
            target_param.data.copy_(_____ )  # self.tau * local_param.data + (1.0 - self.tau) * target_param.data

## Part 6: Training

**TODO 6.1:** Fill the episode loop (use pseudocode from class).

In [ ]:
# Create env and agent
window_size = 30
train_env = TradingEnv(norm_train, window_size)
agent = DQNAgent(window_size + 1, 3, dueling=True)  # Set dueling=True for bonus

episodes = 500
rewards = []

for ep in range(episodes):
    state, _ = train_env.reset()
    total_reward = 0
    done = False
    
    # TODO 6.1: Episode loop (hint: while not done: action = agent.act(state); next_state, reward, done... = env.step(action); agent.step(...); total_reward += reward)
    while not done:
        action = _____  # agent.act(state)
        next_state, reward, done, _, _ = _____  # train_env.step(action)
        _____  # agent.step(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
    
    rewards.append(total_reward)
    print(f'Episode {ep+1}: Reward {total_reward:.2f}')

# Plot training rewards
plt.plot(rewards)
plt.title('Training Rewards')
plt.show()

## Part 7: Evaluation

Test on unseen data. Compare to buy-hold.

**TODO 7.1:** Implement buy-hold baseline.

In [ ]:
test_env = TradingEnv(norm_test, window_size)
state, _ = test_env.reset()
done = False
portfolio = []

while not done:
    action = agent.act(state)
    next_state, reward, done, _, _ = test_env.step(action)
    state = next_state
    portfolio.append(test_env.portfolio)

# TODO 7.1: Buy-hold baseline (start with 10000, buy at first price, value at each step = 10000 * (current / first))
buy_hold = [10000]
first_price = test_prices[window_size]
for p in test_prices[window_size+1:]:
    buy_hold.append(_____ )  # 10000 * (p / first_price)

plt.plot(portfolio, label='DQN')
plt.plot(buy_hold, label='Buy-Hold')
plt.legend()
plt.title('Test Portfolio Value')
plt.show()

## Part 8: Reflection Questions

1. How does DQN perform vs. buy-hold? Why?
2. Try without dueling – does training change?
3. Limitations for real trading? (e.g., fees, multi-stocks)
4. Bonus: Add DDQN (modify targets to use online for argmax, target for value).